In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, StandardScaler, MinMaxScaler
from sklearn.model_selection import train_test_split
import seaborn as sns
import matplotlib.pyplot as plt

# -------------------------
# Load Dataset
# -------------------------
df = pd.read_csv("customer_churn.csv")

print("Dataset Shape:", df.shape)
print(df.head())

# -------------------------
# Handle Missing Values
# -------------------------
df = df.dropna()

# -------------------------
# Encode Categorical Data
# -------------------------
label_encoder = LabelEncoder()

df['PaperlessBilling'] = label_encoder.fit_transform(df['PaperlessBilling'])
df['PaymentMethod'] = label_encoder.fit_transform(df['PaymentMethod'])
df['Contract'] = label_encoder.fit_transform(df['Contract'])

# -------------------------
# Feature Engineering
# -------------------------

# 1 Customer Lifetime Value
df['CustomerLifetimeValue'] = df['MonthlyCharges'] * df['Tenure']

# 2 Average Charges Per Month
df['AvgChargePerMonth'] = df['TotalCharges'] / (df['Tenure'] + 1)

# 3 Payment Efficiency
df['PaymentEfficiency'] = df['TotalCharges'] / (df['MonthlyCharges'] + 1)

# 4 Tenure Group
df['TenureGroup'] = pd.cut(df['Tenure'],
                           bins=[0,12,24,48,60,100],
                           labels=[1,2,3,4,5])

# 5 Charge Ratio
df['ChargeRatio'] = df['MonthlyCharges'] / (df['TotalCharges'] + 1)

# -------------------------
# Outlier Detection (IQR)
# -------------------------
Q1 = df['MonthlyCharges'].quantile(0.25)
Q3 = df['MonthlyCharges'].quantile(0.75)

IQR = Q3 - Q1

df = df[(df['MonthlyCharges'] >= Q1 - 1.5 * IQR) &
        (df['MonthlyCharges'] <= Q3 + 1.5 * IQR)]

# -------------------------
# Feature Scaling
# -------------------------
scaler_standard = StandardScaler()
scaler_minmax = MinMaxScaler()

df[['MonthlyCharges', 'TotalCharges']] = scaler_standard.fit_transform(
    df[['MonthlyCharges', 'TotalCharges']]
)

df[['Tenure']] = scaler_minmax.fit_transform(df[['Tenure']])

# -------------------------
# Correlation Heatmap
# -------------------------
numeric_df = df.select_dtypes(include=['number'])

correlation = numeric_df.corr()

plt.figure(figsize=(10,6))
sns.heatmap(correlation, cmap="coolwarm")
plt.title("Feature Correlation Heatmap")
plt.show()

# -------------------------
# Prepare Dataset
# -------------------------
X = df.drop(["Churn", "CustomerID"], axis=1)
y = df["Churn"]

# -------------------------
# Train Test Split
# -------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Training shape:", X_train.shape)
print("Testing shape:", X_test.shape)

print("Preprocessing Pipeline Completed Successfully")

Dataset Shape: (500, 9)
  CustomerID  Tenure  MonthlyCharges  TotalCharges        Contract  \
0     C00001       6              64          1540        One year   
1     C00002      21             113          1753  Month-to-month   
2     C00003      27              31          1455        Two year   
3     C00004      53              29          7150  Month-to-month   
4     C00005      16             185          1023        One year   

      PaymentMethod PaperlessBilling  SeniorCitizen  Churn  
0       Credit Card               No              1      0  
1  Electronic Check              Yes              1      0  
2       Credit Card               No              1      0  
3  Electronic Check               No              1      0  
4  Electronic Check               No              1      0  


ValueError: could not convert string to float: 'C00001'